LangSmith 추적을 위하여 초기화 합니다.


In [7]:
# ── 환경 변수 로드 ──────────────────────────────────────────
from dotenv import load_dotenv

load_dotenv(r"C:\Users\pc\Desktop\src\.env")


True

In [8]:
# ── LangSmith 추적 설정 ──────────────────────────────────────
# !pip install langchain-teddynote
from langchain_teddynote import logging

logging.langsmith("CH15-Debate-Agent")


LangSmith 추적을 시작합니다.
[프로젝트명]
CH15-Debate-Agent


In [9]:
# ── DialogueAgent: 대화 참여자 기본 클래스 ───────────────────
# - send(): 메시지 히스토리를 LLM에 전달해 다음 발언을 생성
# - receive(): 다른 에이전트의 발언을 히스토리에 추가
# [변경점] model([...]) → model.invoke([...]) (langchain 1.x)
from typing import Callable, List
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


class DialogueAgent:
    def __init__(self, name: str, system_message: SystemMessage, model: ChatOpenAI) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        # model.invoke()로 호출 (구버전: model([...]))
        message = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content="\n".join([self.prefix] + self.message_history)),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        self.message_history.append(f"{name}: {message}")


In [10]:
# ── DialogueSimulator: 다중 에이전트 대화 진행 관리자 ────────
# - inject(): 사회자가 첫 주제를 던지는 역할
# - step(): 다음 발언자를 선택하고 발언을 진행
# - select_next_speaker 함수로 발언 순서를 유연하게 제어 가능
class DialogueSimulator:
    def __init__(self, agents: List[DialogueAgent], selection_function: Callable[[int, List[DialogueAgent]], int]) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        for agent in self.agents:
            agent.receive(name, message)
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. 다음 발언자 선택
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]
        # 2. 발언 생성
        message = speaker.send()
        # 3. 모든 에이전트에 발언 전달
        for receiver in self.agents:
            receiver.receive(speaker.name, message)
        self._step += 1
        return speaker.name, message


In [11]:
# ── DialogueAgentWithTools: 도구를 사용하는 토론 에이전트 ─────
# DialogueAgent를 상속받아 send()에서 LangGraph 에이전트를 실행합니다.
# [변경점] AgentExecutor + create_openai_tools_agent → create_react_agent (LangGraph)
#
# 동작 흐름:
#   1) 메시지 히스토리 + 시스템 메시지를 하나의 프롬프트로 조합
#   2) LangGraph 에이전트가 필요하면 도구(retriever/search)를 호출
#   3) 최종 답변을 반환
from langgraph.prebuilt import create_react_agent


class DialogueAgentWithTools(DialogueAgent):
    def __init__(self, name: str, system_message: SystemMessage, model: ChatOpenAI, tools) -> None:
        super().__init__(name, system_message, model)
        self.tools = tools

    def send(self) -> str:
        # LangGraph 에이전트 생성 (매 발언마다 새로 생성 - 상태 공유 불필요)
        agent = create_react_agent(self.model, self.tools)
        input_text = "\n".join(
            [self.system_message.content] + [self.prefix] + self.message_history
        )
        result = agent.invoke({"messages": [HumanMessage(content=input_text)]})
        content = result["messages"][-1].content
        return AIMessage(content=content).content


In [12]:
# ── 텍스트 파일 로드 → 벡터 스토어 → Retriever 생성 ──────────
# 의사협회측, 정부측 각각 별도 문서로 Retriever를 만들어
# 각 에이전트가 자기 측 근거 자료만 검색하도록 설정합니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader

loader1 = TextLoader("data/의대증원반대.txt", encoding="utf-8")
loader2 = TextLoader("data/의대증원찬성.txt", encoding="utf-8")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

docs1 = loader1.load_and_split(text_splitter)
docs2 = loader2.load_and_split(text_splitter)

vector1 = FAISS.from_documents(docs1, OpenAIEmbeddings())
vector2 = FAISS.from_documents(docs2, OpenAIEmbeddings())

doctor_retriever = vector1.as_retriever(search_kwargs={"k": 5})
gov_retriever = vector2.as_retriever(search_kwargs={"k": 5})


In [13]:
# ── Retriever를 에이전트 도구로 변환 ──────────────────────────
# 각 에이전트에게 자기 측 입장 문서만 검색하는 도구를 부여합니다.
# description이 에이전트가 도구를 선택하는 기준입니다.
from langchain_core.tools.retriever import create_retriever_tool

doctor_retriever_tool = create_retriever_tool(
    doctor_retriever,
    name="document_search",
    description="This is a document about the Korean Medical Association's opposition to the expansion of university medical schools. "
    "Refer to this document when you want to present a rebuttal to the proponents of medical school expansion.",
)

gov_retriever_tool = create_retriever_tool(
    gov_retriever,
    name="document_search",
    description="This is a document about the Korean government's support for the expansion of university medical schools. "
    "Refer to this document when you want to provide a rebuttal to the opposition to medical school expansion.",
)


In [14]:
# ── Tavily 웹 검색 도구 ───────────────────────────────────────
# 문서에 없는 최신 정보를 검색할 때 사용합니다.
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(k=6)
print("검색 도구 준비 완료")


import os
if not os.path.exists("tmp"):
    os.mkdir("tmp")


검색 도구 준비 완료


C:\Users\pc\AppData\Local\Temp\ipykernel_9932\3018377427.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(k=6)


### ① 문서에 기반한 도구


In [15]:
# ── 에이전트별 도구 설정 (문서 검색 방식) ────────────────────
# 각 에이전트가 자기 측 입장 문서 retriever만 사용합니다.
names = {
    "Doctor Union(의사협회)": [doctor_retriever_tool],
    "Government(대한민국 정부)": [gov_retriever_tool],
}

topic = "2024 의정, 대한민국 의대생 정원 증가 정책은 필요한가?"
word_limit = 50
print(f"토론 주제: {topic}")

import os
if not os.path.exists("tmp"):
    os.mkdir("tmp")


토론 주제: 2024 의정, 대한민국 의대생 정원 증가 정책은 필요한가?


### ② 검색(Search) 기반 도구


In [16]:
# ── 에이전트별 도구 설정 (웹 검색 방식) ─────────────────────
# 두 에이전트 모두 Tavily 웹 검색을 사용합니다.
# 문서 검색 방식과 달리 실시간 인터넷 정보를 활용합니다.
names_search = {
    "Doctor Union(의사협회)": [search],
    "Government(대한민국 정부)": [search],
}
topic = "2024년 의정, 대한민국 의대생 정원 증가 정책은 필요한가?"
word_limit = 50
print(f"토론 주제: {topic}")


토론 주제: 2024년 의정, 대한민국 의대생 정원 증가 정책은 필요한가?


In [17]:
# ── LLM으로 각 에이전트 캐릭터 설명 자동 생성 ───────────────
# conversation_description을 LLM에 전달해
# 각 에이전트(의사협회, 정부)의 역할과 관점을 자동으로 생성합니다.
# [변경점] ChatOpenAI()(messages) → ChatOpenAI().invoke(messages)
conversation_description = f"""Here is the topic of conversation: {topic}
The participants are: {', '.join(names.keys())}"""

agent_descriptor_system_message = SystemMessage(
    content="You can add detail to the description of the conversation participant."
)


def generate_agent_description(name):
    agent_specifier_prompt = [
        agent_descriptor_system_message,
        HumanMessage(
            content=f"""{conversation_description}
            Please reply with a description of {name}, in {word_limit} words or less in expert tone. 
            Speak directly to {name}.
            Give them a point of view.
            Do not add anything else. Answer in KOREAN."""
        ),
    ]
    # invoke()로 호출 (구버전: model([...]))
    agent_description = ChatOpenAI(temperature=0).invoke(agent_specifier_prompt).content
    return agent_description


agent_descriptions = {name: generate_agent_description(name) for name in names}
agent_descriptions


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'Doctor Union(의사협회)': '의사협회는 의료계의 전문가들로 구성된 단체로, 의사들의 권익을 보호하고 의료정책에 대한 의견을 제시합니다. 의사들의 입장을 대변하며 환자 안전과 의료진의 복지를 중시합니다. 2024년 의정, 대한민국 의대생 정원 증가 정책의 필요성을 신중히 고려해 주시기 바랍니다.',
 'Government(대한민국 정부)': '대한민국 정부는 국가의 행정을 책임지는 주체로서, 국민의 복지와 발전을 책임져야 합니다. 의대생 정원 증가 정책은 의료 인력 양적 부족 문제를 해결할 수 있는 중요한 정책입니다. 국가 발전과 국민 건강을 고려하여 적극적으로 검토해주시기 바랍니다.'}

In [18]:
agent_descriptions = {
    "Doctor Union(의사협회)": "의사협회는 의료계의 권익을 보호하고 의사들의 이해관계를 대변하는 기관입니다. 의사들의 업무 환경과 안전을 중시하며, 환자 안전과 질 높은 의료 서비스를 제공하기 위해 노력합니다. "
    "지금도 의사의 수는 충분하다는 입장이며, 의대 증원은 필수 의료나 지방 의료 활성화에 대한 실효성이 떨어집니다. 의대 증원을 감행할 경우, 의료 교육 현장의 인프라가 갑작스러운 증원을 감당하지 못할 것이란 우려를 표합니다.",
    "Government(대한민국 정부)": "대한민국 정부는 국가의 행정을 책임지는 주체로서, 국민의 복지와 발전을 책임져야 합니다. "
    "우리나라는 의사수가 절대 부족한 상황이며, 노인인구가 늘어나면서 의료 수요가 급증하고 있습니다. OECD 국가들도 최근 의사수를 늘렸습니다. 또한, 증원된 의사 인력이 필수의료와 지역 의료로 갈 수있도록 튼튼한 의료사고 안정망 구축 및 보상 체계의 공정성을 높이고자 합니다.",
}

In [19]:
# ── 각 에이전트의 시스템 메시지 생성 ────────────────────────
# 에이전트가 토론에서 지켜야 할 규칙을 시스템 메시지로 정의합니다.
# - 도구로 근거를 찾고 출처를 인용해야 함
# - 이미 나온 말을 반복하지 않아야 함
# - 자기 관점에서의 발언이 끝나면 즉시 멈춰야 함
def generate_system_message(name, description, tools):
    return f"""{conversation_description}
    
Your name is {name}.

Your description is as follows: {description}

Your goal is to persuade your conversation partner of your point of view.

DO look up information with your tool to refute your partner's claims.
DO cite your sources.

DO NOT fabricate fake citations.
DO NOT cite any source that you did not look up.

DO NOT restate something that has already been said in the past.
DO NOT add anything else.

Stop speaking the moment you finish speaking from your perspective.

Answer in KOREAN.
"""


agent_system_messages = {
    name: generate_system_message(name, description, tools)
    for (name, tools), description in zip(names.items(), agent_descriptions.values())
}
print("시스템 메시지 생성 완료")


시스템 메시지 생성 완료


In [20]:
# 에이전트 시스템 메시지를 순회합니다.
for name, system_message in agent_system_messages.items():
    # 에이전트의 이름을 출력합니다.
    print(name)
    # 에이전트의 시스템 메시지를 출력합니다.
    print(system_message)

Doctor Union(의사협회)
Here is the topic of conversation: 2024년 의정, 대한민국 의대생 정원 증가 정책은 필요한가?
The participants are: Doctor Union(의사협회), Government(대한민국 정부)

Your name is Doctor Union(의사협회).

Your description is as follows: 의사협회는 의료계의 권익을 보호하고 의사들의 이해관계를 대변하는 기관입니다. 의사들의 업무 환경과 안전을 중시하며, 환자 안전과 질 높은 의료 서비스를 제공하기 위해 노력합니다. 지금도 의사의 수는 충분하다는 입장이며, 의대 증원은 필수 의료나 지방 의료 활성화에 대한 실효성이 떨어집니다. 의대 증원을 감행할 경우, 의료 교육 현장의 인프라가 갑작스러운 증원을 감당하지 못할 것이란 우려를 표합니다.

Your goal is to persuade your conversation partner of your point of view.

DO look up information with your tool to refute your partner's claims.
DO cite your sources.

DO NOT fabricate fake citations.
DO NOT cite any source that you did not look up.

DO NOT restate something that has already been said in the past.
DO NOT add anything else.

Stop speaking the moment you finish speaking from your perspective.

Answer in KOREAN.

Government(대한민국 정부)
Here is the topic of conversation: 2024년 의정, 대한민국 의대생 정원 증가 정책은 필요한가?
The participants are: Doctor Union

`topic_specifier_prompt`를 정의하여 주어진 주제를 더 구체화하는 프롬프트를 생성합니다.

- `temperature` 를 조절하여 더 다양한 주제를 생성할 수 있습니다.


In [21]:
# ── 토론 주제를 더 구체적으로 만들기 ────────────────────────
# LLM이 사회자 역할을 해서 추상적인 주제를 구체적 질문으로 변환합니다.
# temperature=1.0: 더 창의적이고 다양한 표현 사용
# [변경점] ChatOpenAI(temperature=1.0)([...]) → .invoke([...])
topic_specifier_prompt = [
    SystemMessage(content="You can make a topic more specific."),
    HumanMessage(
        content=f"""{topic}
        
        You are the moderator. 
        Please make the topic more specific.
        Please reply with the specified quest in 100 words or less.
        Speak directly to the participants: {*names,}.  
        Do not add anything else.
        Answer in Korean."""
    ),
]
specified_topic = ChatOpenAI(temperature=1.0).invoke(topic_specifier_prompt).content

print(f"Original topic:\n{topic}\n")
print(f"Detailed topic:\n{specified_topic}\n")


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Original topic:
2024년 의정, 대한민국 의대생 정원 증가 정책은 필요한가?

Detailed topic:
의사협회와 대한민국 정부에게 매년 증가하는 의대생 정원이 의학 교육과 의료 서비스 향상에 긍정적인 영향을 미치는지에 대해 의논해보겠습니까? 현재의정 상황과 의학계 인력 양극화 등을 고려하여 2024년 의정, 대한민국 의대생 정원 증가 정책이 필요한지 검토하고 발의된다면 어떻게 실행해야 할지를 논의해 주세요.



혹은 아래와 같이 직접 지정할 수 있습니다.


In [22]:
# 직접 세부 주제 설정
specified_topic = "정부는 2025년 입시부터 의대 입학정원을 2000명 늘린다고 발표했습니다. 이에 의사단체는 전국에서 규탄집회를 열어 반발하고 있습니다. 의대 정원 확대를 둘러싼 논란 쟁점을 짚어보고, 필수 의료와 지역 의료 해법에 대해서 토론해주세요."

In [23]:
# ── 에이전트 인스턴스 생성 ────────────────────────────────────
# 문서 검색용 에이전트 + 웹 검색용 에이전트를 각각 생성합니다.
# gpt-4o 모델 사용 (gpt-4-turbo-preview → gpt-4o로 변경)
agents = [
    DialogueAgentWithTools(
        name=name,
        system_message=SystemMessage(content=system_message),
        model=ChatOpenAI(model_name="gpt-4o", temperature=0.2),
        tools=tools,
    )
    for (name, tools), system_message in zip(
        names.items(), agent_system_messages.values()
    )
]

agents_with_search = [
    DialogueAgentWithTools(
        name=name,
        system_message=SystemMessage(content=system_message),
        model=ChatOpenAI(model_name="gpt-4o", temperature=0.2),
        tools=tools,
    )
    for (name, tools), system_message in zip(
        names_search.items(), agent_system_messages.values()
    )
]

print(f"에이전트 생성 완료: 총 {len(agents) + len(agents_with_search)}개")


에이전트 생성 완료: 총 4개


In [24]:
# ── 발언자 선택 함수 ──────────────────────────────────────────
# step을 에이전트 수로 나눈 나머지로 순서대로 돌아가며 발언합니다.
# ex) 에이전트 2명: 0→0번, 1→1번, 2→0번, 3→1번 ...
def select_next_speaker(step: int, agents: List[DialogueAgent]) -> int:
    return step % len(agents)


In [25]:
# ── 토론 실행 (웹 검색 방식) ──────────────────────────────────
# max_iters=6: 각 에이전트가 3번씩 발언 (총 6턴)
# inject()로 사회자가 주제를 던지고, step()으로 순서대로 발언을 진행합니다.
max_iters = 6
n = 0

simulator = DialogueSimulator(
    agents=agents_with_search, selection_function=select_next_speaker
)
simulator.reset()
simulator.inject("Moderator", specified_topic)

print(f"(Moderator): {specified_topic}\n\n")

while n < max_iters:
    name, message = simulator.step()
    print(f"({name}): {message}\n\n")
    n += 1


(Moderator): 정부는 2025년 입시부터 의대 입학정원을 2000명 늘린다고 발표했습니다. 이에 의사단체는 전국에서 규탄집회를 열어 반발하고 있습니다. 의대 정원 확대를 둘러싼 논란 쟁점을 짚어보고, 필수 의료와 지역 의료 해법에 대해서 토론해주세요.




C:\Users\pc\AppData\Local\Temp\ipykernel_9932\3964627503.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(self.model, self.tools)
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


(Government(대한민국 정부)): Government(대한민국 정부): 대한민국은 현재 의사 수가 절대적으로 부족한 상황입니다. 특히 노인 인구가 증가하면서 의료 수요는 급증하고 있습니다. OECD 국가들도 최근 몇 년간 의사 수를 늘리는 방향으로 정책을 수정하고 있습니다. 이는 의료 서비스의 질을 높이고 국민의 건강을 보장하기 위한 필수적인 조치입니다. 

또한, 증원된 의사 인력이 필수의료와 지역 의료 분야에 종사할 수 있도록, 정부는 의료사고에 대한 안정망을 구축하고 보상 체계의 공정성을 높이기 위한 정책을 추진하고 있습니다. 이를 통해 의료 서비스의 지역 간 불균형을 해소하고, 국민 모두가 양질의 의료 서비스를 받을 수 있도록 하는 것이 목표입니다. 

의사협회 측에서도 이러한 점을 고려하여, 국민의 건강과 복지를 위한 방향으로 협력해 주시기를 바랍니다.




Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


(Doctor Union(의사협회)): 의사협회는 의사 수 부족에 대한 정부의 주장이 과장되었다고 생각합니다. 최근 보건복지부의 토론회에서 제시된 자료에 따르면, 2033년까지 의사 부족은 없을 것으로 예상됩니다. 오히려 의사 과잉이 우려되는 상황입니다. [출처: 경기메디뉴스](https://www.ggmedinews.com/news/articleView.html?idxno=6205)

의대 정원을 늘리는 것만으로는 필수 의료와 지역 의료 문제를 해결할 수 없습니다. 정부가 제안하는 정책은 비과학적 추계에 기반하고 있으며, 이는 정책 실패로 이어질 수 있습니다. 의사 수급 문제는 지역 단위와 전문과목별로 세심하게 접근해야 하며, 단순히 숫자를 늘리는 것이 아니라 실질적인 의료 서비스 개선을 위한 정책이 필요합니다. 

따라서, 의대 정원 증가는 현재의 문제를 해결하는 데 실효성이 떨어지며, 의료 교육 현장의 인프라가 이를 감당할 준비가 되어 있지 않다는 점을 강조하고 싶습니다.




...

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


(Government(대한민국 정부)): 현재 대한민국은 의사 수가 부족한 상황에 직면해 있습니다. 특히 소아과와 산부인과 같은 필수 의료 분야에서 의사 부족이 심각하며, 지방 공공 의료 기관에서는 의사가 없어 비어 있는 진료과가 많습니다. 이는 의료 서비스의 질 저하와 국민 건강에 직접적인 영향을 미칠 수 있습니다. [출처: 조선일보](https://www.chosun.com/national/welfare-medical/2024/02/08/YZAWO4QLRFFI5DRCBCJNGEFUEA/)

따라서 의대 정원 확대는 단순히 의사 수를 늘리는 것이 아니라, 필수 의료와 지역 의료 분야의 인력 부족 문제를 해결하기 위한 필수적인 조치입니다. 정부는 이러한 문제를 해결하기 위해 의료사고 안정망 구축 및 보상 체계의 공정성을 높이는 정책을 추진하고 있습니다. 이는 지역 간 의료 서비스의 불균형을 해소하고, 국민 모두가 양질의 의료 서비스를 받을 수 있도록 하는 데 기여할 것입니다.




Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


(Doctor Union(의사협회)): 의사협회는 정부의 의대 정원 확대 정책이 실질적인 문제 해결에 기여하지 못할 것이라고 주장합니다. 현재 한국의 의사 수는 미국과 일본과 같은 유사한 의료 체계를 가진 국가들과 비슷한 수준이며, 의료 접근성과 진료 횟수는 OECD 상위권에 속합니다. 이는 단순히 의사 수의 부족이 아닌 의료 시스템의 구조적 문제라는 점을 시사합니다. [출처: 의사신문](http://www.doctorstimes.com/news/articleView.html?idxno=230650)

또한, 의사 수 부족 문제는 지역 간, 부문 간 불균형에서 비롯된 것이며, 단순히 의대 정원을 늘리는 것으로 해결될 수 없습니다. 의료계는 전공의들의 과도한 근무 시간, 의료사고의 형사처벌화, 필수의료과목의 저수가 문제를 우선 해결해야 한다고 주장합니다. [출처: 위키백과](https://ko.wikipedia.org/wiki/2024%EB%85%84_%EB%8C%80%ED%95%9C%EB%AF%BC%EA%B5%AD_%EC%9D%98%EC%A0%95_%EA%B0%88%EB%93%B1)

따라서, 의대 정원 증가는 현재의 문제를 해결하는 데 실효성이 떨어지며, 의료 교육 현장의 인프라가 이를 감당할 준비가 되어 있지 않다는 점을 강조하고 싶습니다. [출처: 나무위키](https://namu.wiki/w/2024%EB%85%84%20%EC%9D%98%EB%A3%8C%EC%A0%95%EC%B1%85%20%EC%B6%94%EC%A7%84%20%EB%B0%98%EB%8C%80%20%EC%A7%91%EB%8B%A8%ED%96%89%EB%8F%99)




Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
